# LLM Firewall, Day 3b: fine-tune a multilingual attack detector

**What this notebook does:** takes a model that already understands many languages
(XLM-RoBERTa, MuRIL) and teaches it one extra skill: *is this message an attack (1) or safe (0)?*
That is called **fine-tuning**. Afterwards it scores the model on the test set with the **same tables**
as the Day 3a baselines, so you can compare fairly.

**Before you run (3 steps, about 3 minutes):**
1. Top menu: **Runtime > Change runtime type > T4 GPU > Save**.
2. Open the **Files** panel (folder icon on the left) and drag in these files from
   `llm-firewall\data\processed`: **train.csv, val.csv, test.csv**.
   Optional but nice: also drag in `llm-firewall\results\baselines.csv` so the final table includes the baselines.
3. **Runtime > Run all.** Takes about 15 to 25 minutes. When Google asks to connect to your Drive, click allow
   (that is where the trained model is saved).

Nothing here changes your files. If something breaks, copy the red error text and send it to me.

In [ ]:
!pip -q install sentencepiece protobuf

import copy, gc, math, os, random, shutil, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (SLOW!)")
if device == "cpu":
    print("Go to Runtime > Change runtime type > T4 GPU, then Run all again.")

## 1. Settings

You can change these, but the defaults are chosen so the run finishes in one sitting.
We fixed them **before** looking at any results. Don't tune them against the test numbers.

In [ ]:
MODELS = [
    "xlm-roberta-base",         # general multilingual model (100 languages)
    "google/muril-base-cased",  # trained on Indian languages, including Hinglish-style (romanised) text
    # "microsoft/mdeberta-v3-base",  # optional third model: stronger but about 3x slower here
]

MAX_LEN = 256               # longest message we read, in tokens (word pieces)
EPOCHS = 4                  # passes over the training data
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
THRESHOLD = 0.5             # score >= 0.5 counts as "attack"
LOW_RESOURCE_WEIGHT = 1.0   # later experiment: 3.0 makes Hinglish/Hindi training rows count 3x
SEED = 42

RUN_SEED_REPEATS = False    # set True to retrain the winner with 2 more seeds (+10 min): shows how much results wobble

DATA_DIR = "/content"
OUT_DIR = "/content/outputs"
BEST_DIR = "/content/best_model"
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/llm-firewall"

os.makedirs(OUT_DIR, exist_ok=True)

## 2. Scoring helpers

These are copies of the Day 3a functions (`baselines/metrics.py`), so the tables look the same.

* **recall** = share of real attacks we caught ("caught 13/20")
* **false-alarm rate** = share of *safe* messages we wrongly blocked
* **95% range** = how uncertain the number is, because our Hinglish/Hindi test sets are small

In [ ]:
def wilson(k, n, z=1.96):
    """95% confidence range for a rate k/n (works for small n)."""
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)


def summarize(df, pred_col="pred"):
    y, p = df["label"].to_numpy(), df[pred_col].to_numpy()
    attacks, safe = int((y == 1).sum()), int((y == 0).sum())
    caught = int(((y == 1) & (p == 1)).sum())
    false_alarms = int(((y == 0) & (p == 1)).sum())
    recall = caught / attacks if attacks else float("nan")
    fpr = false_alarms / safe if safe else float("nan")
    precision = caught / (caught + false_alarms) if (caught + false_alarms) else float("nan")
    missed = attacks - caught
    f1 = 2 * caught / (2 * caught + false_alarms + missed) if (2 * caught + false_alarms + missed) else float("nan")
    lo, hi = wilson(caught, attacks)
    return {"rows": len(df), "attacks": attacks, "caught": caught, "recall": recall,
            "recall_ci_low": lo, "recall_ci_high": hi, "safe": safe, "false_alarms": false_alarms,
            "false_alarm_rate": fpr, "precision": precision, "f1": f1}


def report(df, by, pred_col="pred"):
    rows = []
    for key, part in df.groupby(by, sort=False):
        row = {"slice": key if isinstance(key, str) else " / ".join(map(str, key))}
        row.update(summarize(part, pred_col))
        rows.append(row)
    return pd.DataFrame(rows)


def pretty(table):
    out = pd.DataFrame({"slice": table["slice"]})
    out["attacks caught"] = [f"{r.caught}/{r.attacks}" if r.attacks else "-" for r in table.itertuples()]
    out["recall (95% range)"] = [
        f"{r.recall:.0%} ({r.recall_ci_low:.0%}-{r.recall_ci_high:.0%})" if r.attacks else "-"
        for r in table.itertuples()]
    out["safe blocked"] = [f"{r.false_alarms}/{r.safe}" if r.safe else "-" for r in table.itertuples()]
    out["false-alarm rate"] = [f"{r.false_alarm_rate:.1%}" if r.safe else "-" for r in table.itertuples()]
    out["F1"] = [f"{r.f1:.2f}" if (r.attacks and r.safe) else "-" for r in table.itertuples()]
    return out


LOW_RESOURCE = {"hi", "hinglish"}
LANG_ORDER = ["en", "hinglish", "hi", "other"]


def origin_of(source):
    if source == "my_examples.csv":
        return "written by you"
    if source in {"hinglish_seed.csv", "hard_negatives.csv", "my_examples_extra.csv"}:
        return "written by Claude (seed)"
    if source == "generated_review.csv":
        return "machine-translated, you reviewed"
    if source == "generated_indirect":
        return "template-generated"
    return "public dataset"


def slices(df):
    """All the ways we cut the results: (kind of slice, table)."""
    parts = [("overall", report(df.assign(_all="ALL"), "_all"))]
    lang = report(df, "lang")
    lang["_o"] = lang["slice"].map({l: i for i, l in enumerate(LANG_ORDER)})
    parts.append(("language", lang.sort_values("_o").drop(columns="_o")))
    parts.append(("origin (Hinglish+Hindi only)", report(df[df["lang"].isin(LOW_RESOURCE)], "origin")))
    length = pd.cut(df["text"].str.len(), bins=[0, 100, 300, 10**9],
                    labels=["short (under 100 chars)", "medium (100-300)", "long (over 300)"]).astype(str)
    parts.append(("length", report(df.assign(_len=length), "_len")))
    parts.append(("category", report(df, "category")))
    return parts


def evaluate(preds, split):
    """preds: {model name: dataframe with columns label, pred, lang, ...}"""
    rows = []
    for model, df in preds.items():
        for kind, table in slices(df):
            t = table.copy()
            t.insert(0, "slice_type", kind)
            t.insert(0, "split", split)
            t.insert(0, "model", model)
            rows.append(t)
    return pd.concat(rows, ignore_index=True)


def show(results, kind, title):
    part = results[results["slice_type"] == kind]
    p = pretty(part)
    p.insert(0, "model", part["model"].to_numpy())
    print("\n" + title)
    display(p)


def balanced_f1(df):
    """Average F1 over English, Hinglish and Hindi. Used to pick the best epoch/model on the VALIDATION set,
    so English (83% of the data) cannot hide a weak Hinglish or Hindi score."""
    vals = []
    for lang in ["en", "hinglish", "hi"]:
        part = df[df["lang"] == lang]
        if part["label"].nunique() == 2:
            vals.append(summarize(part)["f1"])
    return float(np.mean(vals)) if vals else float("nan")

## 3. Load the data

In [ ]:
def load(split):
    df = pd.read_csv(f"{DATA_DIR}/{split}.csv", encoding="utf-8-sig")
    df["text"] = df["text"].astype(str)
    df["origin"] = df["source"].map(origin_of)
    return df

train, val, test = load("train"), load("val"), load("test")
print(f"train {len(train)} | val {len(val)} | test {len(test)} rows")
print("Test set: attacks (1) and safe (0) per language")
display(pd.crosstab(test["lang"], test["label"]))

## 4. The training code

Read this once; it is the heart of the notebook. In plain words, for every batch of 16 messages:

1. the model reads them and gives a guess (attack or safe),
2. we measure how wrong it was (the **loss**),
3. we nudge the model's numbers a tiny bit so it would be less wrong next time.

After each full pass (**epoch**) we check the **validation** set and remember the best version.
Attacks are rarer than safe messages, so a mistake on an attack counts more (**class weight**).

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def make_loader(df, tok, shuffle, use_weights=False):
    enc = tok(df["text"].tolist(), truncation=True, max_length=MAX_LEN)
    w = np.where(df["lang"].isin(LOW_RESOURCE), LOW_RESOURCE_WEIGHT, 1.0) if use_weights else np.ones(len(df))
    items = [{"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i],
              "labels": int(df["label"].iloc[i]), "w": float(w[i])} for i in range(len(df))]

    def collate(batch):
        padded = tok.pad([{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch],
                         return_tensors="pt")
        padded["labels"] = torch.tensor([b["labels"] for b in batch])
        padded["w"] = torch.tensor([b["w"] for b in batch], dtype=torch.float)
        return padded

    return DataLoader(items, batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=collate)


@torch.no_grad()
def predict_scores(model, loader, use_amp):
    """Returns P(attack) for every row, in the same order as the dataframe."""
    model.eval()
    out = []
    for batch in loader:
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            logits = model(input_ids=batch["input_ids"].to(device),
                           attention_mask=batch["attention_mask"].to(device)).logits
        out.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def with_preds(df, scores):
    d = df.copy()
    d["score"] = scores
    d["pred"] = (d["score"] >= THRESHOLD).astype(int)
    return d


def train_one(model_name, seed):
    """Fine-tune one model. Returns its val/test predictions and the best model (on CPU)."""
    t0 = time.time()
    set_seed(seed)
    # fp16 speeds things up on the T4 GPU; mDeBERTa can become unstable in fp16, so it runs in full precision.
    use_amp = (device == "cuda") and ("mdeberta" not in model_name.lower())

    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

    train_loader = make_loader(train, tok, shuffle=True, use_weights=True)
    val_loader = make_loader(val, tok, shuffle=False)
    test_loader = make_loader(test, tok, shuffle=False)

    n_attack = int(train["label"].sum()); n_safe = len(train) - n_attack
    class_w = torch.tensor([1.0, n_safe / n_attack], device=device)   # attacks count ~2x

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    scaler = torch.amp.GradScaler(device, enabled=use_amp)

    best_score, best_epoch, best_state = -1.0, 0, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=batch["input_ids"].to(device),
                               attention_mask=batch["attention_mask"].to(device)).logits
            labels = batch["labels"].to(device)
            per_row = torch.nn.functional.cross_entropy(logits.float(), labels, reduction="none")
            loss = (per_row * class_w[labels] * batch["w"].to(device)).mean()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item()

        val_df = with_preds(val, predict_scores(model, val_loader, use_amp))
        score = balanced_f1(val_df)
        low = summarize(val_df[val_df["lang"].isin(LOW_RESOURCE)])
        print(f"  epoch {epoch}/{EPOCHS}  train loss {running / len(train_loader):.3f}  |  validation: "
              f"balanced F1 {score:.3f}, Hinglish+Hindi caught {low['caught']}/{low['attacks']}, "
              f"safe blocked {low['false_alarms']}/{low['safe']}")
        if score > best_score:
            best_score, best_epoch = score, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)          # go back to the best epoch
    val_df = with_preds(val, predict_scores(model, val_loader, use_amp))
    test_df = with_preds(test, predict_scores(model, test_loader, use_amp))
    print(f"  kept epoch {best_epoch} (validation balanced F1 {best_score:.3f}); took {(time.time() - t0) / 60:.1f} min")
    return {"val": val_df, "test": test_df, "val_score": best_score, "best_epoch": best_epoch,
            "model": model.cpu(), "tokenizer": tok}

## 5. Train and score each model

The winner is chosen by the **validation** score, never by the test score.
The test set is looked at once per model, to report the final numbers.

In [ ]:
val_preds, test_preds, summary = {}, {}, []
best = {"name": None, "val_score": -1.0}

for name in MODELS:
    print(f"\n=== {name} ===")
    out = train_one(name, SEED)
    val_preds[name], test_preds[name] = out["val"], out["test"]
    summary.append({"model": name, "kept epoch": out["best_epoch"], "validation balanced F1": round(out["val_score"], 3)})
    if out["val_score"] > best["val_score"]:
        best = {"name": name, "val_score": out["val_score"]}
        shutil.rmtree(BEST_DIR, ignore_errors=True)
        out["model"].save_pretrained(BEST_DIR)
        out["tokenizer"].save_pretrained(BEST_DIR)
    del out
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

print("\nWinner on validation:", best["name"])
display(pd.DataFrame(summary))

## 6. Results on the TEST set

Read these like the Day 3a tables. The things to look at first:
* **Hinglish** row: Day 3a's TF-IDF baseline caught 13/20 and blocked 4/28 safe messages.
* **short (under 100 chars)** row: the baseline caught 13/19.
* Did the safe-message blocks stay low? A firewall that blocks real customers is useless.

In [ ]:
results = pd.concat([evaluate(test_preds, "test"), evaluate(val_preds, "val")], ignore_index=True)

# add the Day 3a baselines to the same table, if you uploaded results/baselines.csv
base_path = f"{DATA_DIR}/baselines.csv"
if os.path.exists(base_path):
    base = pd.read_csv(base_path, encoding="utf-8-sig")
    results = pd.concat([base[base["model"].isin(["rules", "tfidf_plain"])], results], ignore_index=True)
    print("Included the Day 3a baselines (rules, tfidf_plain) for comparison.")
else:
    print("baselines.csv not uploaded; showing only the fine-tuned models.")

t = results[results["split"] == "test"]
show(t, "overall", "### Overall")
show(t, "language", "### By language")
show(t, "origin (Hinglish+Hindi only)", "### Hinglish + Hindi rows, by where the row came from")
show(t, "length", "### By message length")
show(t, "category", "### By category")

In [ ]:
def scoreboard(results):
    t = results[results["split"] == "test"]
    rows = []
    for model, g in t.groupby("model", sort=False):
        def cell(kind, sl):
            r = g[(g["slice_type"] == kind) & (g["slice"] == sl)]
            if r.empty:
                return "-"
            r = r.iloc[0]
            return f"caught {int(r.caught)}/{int(r.attacks)}, blocked {int(r.false_alarms)}/{int(r.safe)} safe"
        rows.append({"model": model, "overall": cell("overall", "ALL"), "Hinglish": cell("language", "hinglish"),
                     "Hindi": cell("language", "hi"), "short messages": cell("length", "short (under 100 chars)")})
    return pd.DataFrame(rows)

print("SCOREBOARD (test set). Send me a screenshot of this table.")
display(scoreboard(results))

## 7. Save everything

* `finetune_results.csv`: all the numbers (goes into your repo's `results/` folder)
* `errors_test_finetuned.csv`: every test mistake (read the Hinglish/Hindi ones, like on Day 3a)
* `predictions_test.csv`: the raw attack score for every test row (we will use it to choose the cut-off on Day 6)
* the winning model, saved to your Google Drive in the folder `llm-firewall/best_model` (used on Day 4)

In [ ]:
fine = results[results["model"].isin(MODELS)]
fine.to_csv(f"{OUT_DIR}/finetune_results.csv", index=False, encoding="utf-8-sig")

errs, preds_rows = [], []
for model, d in test_preds.items():
    bad = d[d["pred"] != d["label"]].copy()
    bad["error"] = np.where(bad["label"] == 1, "missed attack", "false alarm")
    bad.insert(0, "model", model)
    errs.append(bad[["model", "error", "lang", "origin", "category", "score", "text"]])
    keep = d[["lang", "origin", "category", "label", "score", "pred", "text"]].copy()
    keep.insert(0, "model", model)
    preds_rows.append(keep)
pd.concat(errs).assign(_o=lambda x: x["lang"].map({l: i for i, l in enumerate(LANG_ORDER)})) \
    .sort_values(["_o", "model", "error"]).drop(columns="_o") \
    .to_csv(f"{OUT_DIR}/errors_test_finetuned.csv", index=False, encoding="utf-8-sig")
pd.concat(preds_rows).to_csv(f"{OUT_DIR}/predictions_test.csv", index=False, encoding="utf-8-sig")
print("Saved CSVs in", OUT_DIR, "->", sorted(os.listdir(OUT_DIR)))

if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        shutil.copytree(BEST_DIR, f"{DRIVE_DIR}/best_model", dirs_exist_ok=True)
        for f in os.listdir(OUT_DIR):
            shutil.copy(f"{OUT_DIR}/{f}", f"{DRIVE_DIR}/{f}")
        print("Saved the winning model and the CSVs to your Google Drive folder:", DRIVE_DIR)
    except ImportError:
        print("Not running on Colab: skipped the Drive copy.")

try:
    from google.colab import files
    files.download(f"{OUT_DIR}/finetune_results.csv")
    files.download(f"{OUT_DIR}/errors_test_finetuned.csv")
except ImportError:
    pass

## 8. Optional: how much do the results wobble?

Fine-tuning has randomness. Train the **winner** again with two other seeds and see how far the test numbers move.
With only 20 Hinglish attacks in the test set, a swing of a few attacks is normal. Reporting that range is what makes
your results trustworthy. To run it, set `RUN_SEED_REPEATS = True` in the Settings cell and run this cell.

In [ ]:
if RUN_SEED_REPEATS:
    rows = []
    for s in [SEED, 1, 2]:
        print(f"\n=== {best['name']}  seed {s} ===")
        if s == SEED:
            d = test_preds[best["name"]]
        else:
            out = train_one(best["name"], s)
            d = out["test"]
            del out; gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()
        def cell(part):
            r = summarize(part)
            return f"caught {r['caught']}/{r['attacks']}, blocked {r['false_alarms']}/{r['safe']}"
        rows.append({"seed": s, "overall": cell(d), "Hinglish": cell(d[d['lang'] == 'hinglish']),
                     "Hindi": cell(d[d['lang'] == 'hi']), "short messages": cell(d[d['text'].str.len() <= 100])})
    display(pd.DataFrame(rows))
else:
    print("Skipped (RUN_SEED_REPEATS is False).")

## Done

Send me a screenshot of the **SCOREBOARD** table, and upload `finetune_results.csv` and
`errors_test_finetuned.csv` (they were downloaded automatically; also in your Drive folder `llm-firewall`).